# Perception Stack - Notebook 03: Enhanced LLaVA Inference

This notebook runs the full enhanced condition and all three ablation conditions
of the Perception Stack evaluation. LLaVA-1.5 7B receives the last frame,
object identity, and a structured context string assembled from the outputs of
SAM 2 (Stage 2), Depth Anything V2 (Stage 3), and RAFT (Stage 4).
Four conditions are evaluated in a single loop: full context, no SAM2 context,
no depth context, and no RAFT context.

## Steps
- Mount Google Drive and define all constants, paths, and ablation condition folders
- Install dependencies and load LLaVA-1.5 7B with 4-bit NF4 quantization
- Load selected clip IDs, QA JSON, and verify keyframes exist on Drive
- Define the three context builder functions: SAM 2 context, depth context,
  and RAFT context
- Define the context assembler that toggles stage contributions per condition
  and preview context strings for all four conditions
- Define the answer parser and scorer (identical to Notebook 01)
- Define the prompt builder, LLaVA inference function (new-tokens-only decoding),
  and per-clip processing function
- Run main inference loop: for each clip and each of the four conditions,
  run inference and save per-clip JSON to Drive with per-clip checkpointing
- Aggregate metrics across all clips and conditions; print summary table
- Save per-condition summary CSVs to the analysis folder on Drive

---
> **Note:** This notebook is natively developed and run on Google Colab (A100 GPU required).
> If it does not render correctly in your viewer, please open it directly on Colab:
> [Open in Google Colab](<https://colab.research.google.com/drive/10L1Vk_edI_KGIWp4YEMrPyPlJ1YOjk-j?usp=sharing>)

---
# Cell 1 — Mount Drive + Constants + Paths

In [1]:
# ── Cell 1 — Mount Drive + Constants + Paths ───────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── CLEAR: Delete all NB03 outputs ─────────────────────────────────────────
import os, shutil

BASE = "/content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project"

# Clear all enhanced condition folders
llava_dir = os.path.join(BASE, "outputs/stage4_llava")
conditions = ['full', 'no_sam2', 'no_depth', 'no_raft']
deleted = 0
for condition in conditions:
    folder = os.path.join(llava_dir, condition)
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.startswith("enhanced_") and f.endswith(".json"):
                os.remove(os.path.join(folder, f))
                deleted += 1
print(f"Deleted {deleted} enhanced JSONs")

# Clear enhanced error logs
for f in os.listdir(llava_dir):
    if f.startswith("enhanced_errors_"):
        os.remove(os.path.join(llava_dir, f))
        print(f"Deleted error log: {f}")

# Clear NB03 CSVs
analysis_dir = os.path.join(BASE, "analysis")
deleted_csvs = 0
for f in os.listdir(analysis_dir):
    if f.startswith("enhanced_") and f.endswith(".csv"):
        os.remove(os.path.join(analysis_dir, f))
        deleted_csvs += 1
print(f"Deleted {deleted_csvs} enhanced CSVs")

print("NB03 slate cleared.")

Deleted 8 enhanced JSONs
Deleted 4 enhanced CSVs
NB03 slate cleared.


In [4]:
import os

# ── Constants ──────────────────────────────────────────────────────────────
NO_CLIPS   = 70      # change this one number to scale up
SEED       = 42
MODEL_ID   = "llava-hf/llava-1.5-7b-hf"
HF_TOKEN   = userdata.get('HF_TOKEN')

# Ablation condition prefixes -- controls what goes into context string
ABLATION_CONDITIONS = ['full', 'no_sam2', 'no_depth', 'no_raft']

# ── Base path on Drive ─────────────────────────────────────────────────────
BASE        = "/content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project"
CLIPS_DIR   = os.path.join(BASE, "data/clips")
QA_PATH     = os.path.join(BASE, "data/qa/validation_qa.json")
ANN_DIR     = os.path.join(BASE, "data/annotations")
KF_DIR      = os.path.join(BASE, "data/keyframes")
ID_PATH     = os.path.join(BASE, "data/selected_clip_ids.json")
SAM2_DIR    = os.path.join(BASE, "outputs/stage1_sam2")
DEPTH_DIR   = os.path.join(BASE, "outputs/stage2_depth")
RAFT_DIR    = os.path.join(BASE, "outputs/stage3_raft")
# OUT_DIR     = os.path.join(BASE, "outputs/stage4_llava")
OUT_DIR      = os.path.join(BASE, "outputs/stage4_llava")
OUT_ENHANCED = {
    condition: os.path.join(OUT_DIR, condition)
    for condition in ABLATION_CONDITIONS
}

for path in OUT_ENHANCED.values():
    os.makedirs(path, exist_ok=True)

FIGURES_DIR = os.path.join(BASE, "figures")
ANALYSIS_DIR= os.path.join(BASE, "analysis")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(ANALYSIS_DIR, exist_ok=True)

# ── Load selected clip IDs ─────────────────────────────────────────────────
import json
with open(ID_PATH, 'r') as f:
    ALL_SELECTED_IDS = json.load(f)

SELECTED_IDS = ALL_SELECTED_IDS[:NO_CLIPS]
print(f"Drive mounted.")
print(f"Working with {NO_CLIPS} clips: {SELECTED_IDS}")
print(f"Ablation conditions: {ABLATION_CONDITIONS}")

Drive mounted.
Working with 70 clips: [10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010, 10011, 10012, 10013, 10014, 10015, 10016, 10017, 10018, 10019, 10020, 10021, 10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031, 10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041, 10042, 10043, 10044, 10045, 10046, 10047, 10048, 10049, 10050, 10051, 10052, 10053, 10054, 10055, 10056, 10057, 10058, 10059, 10060, 10061, 10062, 10063, 10064, 10065, 10066, 10067, 10068, 10069, 10070]
Ablation conditions: ['full', 'no_sam2', 'no_depth', 'no_raft']


---

# Cell 2 — Install Dependencies + Load Model

In [5]:
# ── Cell 2 — Install Dependencies + Load Model ────────────────────────────
!pip install -q bitsandbytes transformers accelerate pillow

import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# ── 4-bit quantization config ──────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# ── Load model + processor ─────────────────────────────────────────────────
print("Loading model... (takes ~2 min on first run)")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    token=HF_TOKEN,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model.eval()

print(f"Model loaded on: {next(model.parameters()).device}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.5 MB/s eta 0:00:00
Loading model... (takes ~2 min on first run)


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Model loaded on: cuda:0
VRAM used: 4.06 GB


---

# Cell 3 — Load Clip IDs, QA JSON, Keyframes, Annotations

In [6]:
# ── Cell 3 — Load Clip IDs, QA JSON, Keyframes, Annotations ───────────────
import json
from PIL import Image

# ── Load selected clip IDs ─────────────────────────────────────────────────
with open(ID_PATH, 'r') as f:
    ALL_SELECTED_IDS = json.load(f)

SELECTED_IDS = ALL_SELECTED_IDS[:NO_CLIPS]
print(f"Total IDs on Drive : {len(ALL_SELECTED_IDS)}")
print(f"Using this run     : {len(SELECTED_IDS)}")

# ── Load full QA JSON, index by scene_index ────────────────────────────────
with open(QA_PATH, 'r') as f:
    all_qa = json.load(f)
qa_by_id = {entry['scene_index']: entry for entry in all_qa}
print(f"QA entries loaded  : {len(qa_by_id)}")

# ── Verify keyframes exist ─────────────────────────────────────────────────
missing_kf = [
    vid_id for vid_id in SELECTED_IDS
    if not os.path.exists(os.path.join(KF_DIR, f"frame_{vid_id:05d}.jpg"))
]
if missing_kf:
    print(f"WARNING: Missing keyframes for: {missing_kf}")
else:
    print(f"All {len(SELECTED_IDS)} keyframes found.")

# ── Load object identity from annotations ─────────────────────────────────
# Returns "Color Material Shape" strings per clip, e.g. "Red rubber sphere"
def get_object_identity_list(vid_id):
    """
    Load annotation for a clip and return sorted list of
    human-readable object descriptions matching CLEVRER vocabulary.
    """
    ann_path = os.path.join(ANN_DIR, f"annotation_{vid_id}.json")
    with open(ann_path, 'r') as f:
        ann = json.load(f)
    labels = []
    for obj in ann['object_property']:
        c = obj['color'].capitalize()
        m = obj['material']
        s = obj['shape']
        labels.append(f"{c} {m} {s}")
    return sorted(labels)

# ── Verify annotations load correctly ─────────────────────────────────────
print("\nObject identity check:")
for vid_id in SELECTED_IDS:
    labels = get_object_identity_list(vid_id)
    print(f"  Clip {vid_id}: {labels}")

print("\nCell 3 complete.")

Total IDs on Drive : 70
Using this run     : 70
QA entries loaded  : 5000
All 70 keyframes found.

Object identity check:
  Clip 10001: ['Brown metal cube', 'Gray metal cube', 'Green metal sphere', 'Yellow metal cylinder', 'Yellow rubber sphere']
  Clip 10002: ['Blue metal cylinder', 'Brown metal cylinder', 'Cyan rubber cube', 'Gray metal sphere', 'Gray rubber sphere', 'Green metal cube']
  Clip 10003: ['Blue metal sphere', 'Yellow metal cube', 'Yellow metal cylinder', 'Yellow rubber sphere']
  Clip 10004: ['Blue metal cube', 'Blue rubber cylinder', 'Brown metal sphere', 'Yellow rubber cube']
  Clip 10005: ['Brown rubber sphere', 'Cyan metal sphere', 'Red metal cube', 'Red rubber cylinder', 'Yellow rubber cube']
  Clip 10006: ['Brown metal cube', 'Gray metal sphere', 'Green rubber cube', 'Purple metal cylinder']
  Clip 10007: ['Blue metal cube', 'Brown metal cylinder', 'Gray metal cylinder', 'Gray rubber cube', 'Green rubber sphere', 'Red metal sphere']
  Clip 10008: ['Cyan metal cube'

---
# Cell 4 — Context String Builders

In [7]:
# ── Cell 4 — Context String Builders ──────────────────────────────────────
import numpy as np

# ── Load threshold from Drive ──────────────────────────────────────────────
with open(os.path.join(BASE, "outputs/depth_threshold.json"), 'r') as f:
    DEPTH_THRESHOLD = json.load(f)['depth_collision_threshold']
print(f"Depth threshold loaded: {DEPTH_THRESHOLD}")

# ── SAM 2 context ──────────────────────────────────────────────────────────
def build_sam2_context(vid_id):
    npz = np.load(os.path.join(SAM2_DIR, f"stage1_sam2_{vid_id:05d}.npz"),
                  allow_pickle=True)

    overlap_obj_a  = [str(x) for x in npz['overlap_obj_a']]
    overlap_obj_b  = [str(x) for x in npz['overlap_obj_b']]
    overlap_frames = [int(x) for x in npz['overlap_frames']]
    overlap_pixels = [int(x) for x in npz['overlap_pixels']]

    def short(label):
        parts = label.split('_')
        return f"{parts[0]} {parts[2]}"

    lines = [f"Object overlaps detected: {len(overlap_frames)}"]
    for obj_a, obj_b, frame, pixels in zip(overlap_obj_a, overlap_obj_b,
                                            overlap_frames, overlap_pixels):
        lines.append(f"  {short(obj_a)} <-> {short(obj_b)} "
                     f"first overlap at frame {frame} ({pixels} pixels)")

    return "\n".join(lines)

# ── Depth context ──────────────────────────────────────────────────────────
def build_depth_context(vid_id):
    """
    Builds depth contribution to context string:
    - Depth at overlap frames with same-plane classification
    - Final scene depth at frame 127
    """
    npz = np.load(os.path.join(DEPTH_DIR, f"stage2_depth_{vid_id:05d}.npz"),
                  allow_pickle=True)

    lines = [
        "Depth information (relative estimates within each frame,",
        "0 = closest to camera, 1 = farthest from camera).",
        f"Objects with depth difference below {DEPTH_THRESHOLD} "
        f"are likely on the same collision plane.",
        ""
    ]

    lines.append("Depth at spatial overlap events:")
    n = int(npz['n_overlaps'])
    for i in range(n):
        obj_a   = str(npz[f'overlap_{i}_obj_a']).replace('_', ' ')
        obj_b   = str(npz[f'overlap_{i}_obj_b']).replace('_', ' ')
        frame   = int(npz[f'overlap_{i}_frame'])
        depth_a = float(npz[f'overlap_{i}_depth_a'])
        depth_b = float(npz[f'overlap_{i}_depth_b'])
        diff    = float(npz[f'overlap_{i}_diff'])
        plane   = "same collision plane" if diff < DEPTH_THRESHOLD else "different planes"
        lines.append(f"  - {obj_a.capitalize()} and {obj_b} at frame {frame}: "
                     f"depths {depth_a:.2f} and {depth_b:.2f}, "
                     f"difference {diff:.2f} ({plane}).")

    lines.append("")
    lines.append("Final scene depth (frame 127):")
    last_keys = sorted([k for k in npz.files if k.startswith('last__')])
    for key in last_keys:
        label = key.replace('last__', '').replace('_', ' ')
        depth = float(npz[key])
        lines.append(f"  - {label.capitalize()}: depth {depth:.2f}")

    return "\n".join(lines)


# ── RAFT velocity context ──────────────────────────────────────────────────
def build_raft_context(vid_id):
    """
    Builds RAFT contribution to context string:
    - Pre-overlap velocities for each overlap pair
    - Final velocities at frames 124-127
    """
    raft_npz = np.load(os.path.join(RAFT_DIR, f"stage3_raft_{vid_id:05d}.npz"),
                       allow_pickle=True)
    sam2_npz = np.load(os.path.join(SAM2_DIR, f"stage1_sam2_{vid_id:05d}.npz"),
                       allow_pickle=True)

    obj_labels     = [str(l) for l in sam2_npz['obj_labels']]
    overlap_obj_a  = [str(x) for x in sam2_npz['overlap_obj_a']]
    overlap_obj_b  = [str(x) for x in sam2_npz['overlap_obj_b']]
    overlap_frames = [int(x) for x in sam2_npz['overlap_frames']]

    def direction_str(dx, dy):
        parts = []
        if abs(dx) > 0.3:
            parts.append("rightward" if dx > 0 else "leftward")
        if abs(dy) > 0.3:
            parts.append("downward" if dy > 0 else "upward")
        return " and ".join(parts) if parts else "nearly stationary"

    lines = [
        "Velocity information (pixels per frame, 3-frame gap):",
        "Positive x = rightward, negative x = leftward.",
        "Positive y = downward, negative y = upward.",
        "Note: velocity estimates are approximate. "
        "Readings at some overlap frames may be less reliable due to occlusion.",
        ""
    ]

    lines.append("Velocity just before each spatial overlap event:")
    for obj_a, obj_b, ov_frame in zip(overlap_obj_a, overlap_obj_b, overlap_frames):
        tag  = f"pre_overlap_{obj_a}__{obj_b}"
        ra   = obj_a.replace('_', ' ')
        rb   = obj_b.replace('_', ' ')
        dx_a = raft_npz.get(f"{tag}__{obj_a}__dx")
        dy_a = raft_npz.get(f"{tag}__{obj_a}__dy")
        dx_b = raft_npz.get(f"{tag}__{obj_b}__dx")
        dy_b = raft_npz.get(f"{tag}__{obj_b}__dy")
        if dx_a is None or dx_b is None:
            continue
        dx_a, dy_a = float(dx_a[0]), float(dy_a[0])
        dx_b, dy_b = float(dx_b[0]), float(dy_b[0])
        lines.append(f"  - Overlap at frame {ov_frame}: "
                     f"{ra.capitalize()} moving {direction_str(dx_a, dy_a)} "
                     f"({dx_a:+.2f}, {dy_a:+.2f}), "
                     f"{rb} moving {direction_str(dx_b, dy_b)} "
                     f"({dx_b:+.2f}, {dy_b:+.2f}).")

    lines.append("")
    lines.append("Final object velocities (frames 124 to 127):")
    for label in obj_labels:
        key_dx = f"final__{label}__dx"
        key_dy = f"final__{label}__dy"
        if key_dx not in raft_npz:
            continue
        dx = float(raft_npz[key_dx][0])
        dy = float(raft_npz[key_dy][0])
        readable = label.replace('_', ' ')
        lines.append(f"  - {readable.capitalize()}: "
                     f"moving {direction_str(dx, dy)} ({dx:+.2f}, {dy:+.2f}).")

    return "\n".join(lines)

Depth threshold loaded: 0.08


In [8]:
# ── Preview for clip 10205 ─────────────────────────────────────────────────
print("── SAM2 context preview (clip 10205) ────────────────────────")
print(build_sam2_context(10001))
print("\n── Depth context preview (clip 10205) ───────────────────────")
print(build_depth_context(10001))
print("\n── RAFT context preview (clip 10205) ────────────────────────")
print(build_raft_context(10001))
print("\nCell 4 complete.")

── SAM2 context preview (clip 10205) ────────────────────────
Object overlaps detected: 2
  gray cube <-> green sphere first overlap at frame 99 (1 pixels)
  green sphere <-> yellow cylinder first overlap at frame 92 (1 pixels)

── Depth context preview (clip 10205) ───────────────────────
Depth information (relative estimates within each frame,
0 = closest to camera, 1 = farthest from camera).
Objects with depth difference below 0.08 are likely on the same collision plane.

Depth at spatial overlap events:
  - Gray metal cube and green metal sphere at frame 99: depths 0.25 and 0.26, difference 0.01 (same collision plane).
  - Green metal sphere and yellow metal cylinder at frame 92: depths 0.29 and 0.34, difference 0.05 (same collision plane).

Final scene depth (frame 127):
  - Brown metal cube: depth 0.58
  - Gray metal cube: depth 0.25
  - Green metal sphere: depth 0.27
  - Yellow metal cylinder: depth 0.33
  - Yellow rubber sphere: depth 0.30

── RAFT context preview (clip 10205) 

---
# Cell 5 — Assemble Full Context String

In [9]:
# ── Cell 5 — Assemble Full Context String ─────────────────────────────────

def assemble_context(vid_id, condition='full'):
    """
    Assemble the full context string for a clip based on ablation condition.
    condition: 'full' | 'no_sam2' | 'no_depth' | 'no_raft'

    Object identity is always included in all conditions.
    """
    # ── Object identity (always included) ─────────────────────────────────
    obj_labels = get_object_identity_list(vid_id)
    lines = ["Objects in this scene:"]
    for label in obj_labels:
        lines.append(f"  - {label}")
    lines.append("")

    # ── Stage contributions (toggled by condition) ─────────────────────────
    if condition != 'no_sam2':
        lines.append(build_sam2_context(vid_id))
        lines.append("")

    if condition != 'no_depth':
        lines.append(build_depth_context(vid_id))
        lines.append("")

    if condition != 'no_raft':
        lines.append(build_raft_context(vid_id))
        lines.append("")

    return "\n".join(lines).strip()


# ── Preview all 4 conditions for clip 10205 ───────────────────────────────
for condition in ABLATION_CONDITIONS:
    ctx = assemble_context(10001, condition)
    n_chars = len(ctx)
    n_lines = ctx.count('\n') + 1
    print(f"── Condition: {condition} | {n_lines} lines | {n_chars} chars ──")
    print(ctx[:300])
    print("  ...")
    print()

print("Cell 5 complete.")

── Condition: full | 41 lines | 1967 chars ──
Objects in this scene:
  - Brown metal cube
  - Gray metal cube
  - Green metal sphere
  - Yellow metal cylinder
  - Yellow rubber sphere

Object overlaps detected: 2
  gray cube <-> green sphere first overlap at frame 99 (1 pixels)
  green sphere <-> yellow cylinder first overlap at frame 92 (1 pix
  ...

── Condition: no_sam2 | 37 lines | 1800 chars ──
Objects in this scene:
  - Brown metal cube
  - Gray metal cube
  - Green metal sphere
  - Yellow metal cylinder
  - Yellow rubber sphere

Depth information (relative estimates within each frame,
0 = closest to camera, 1 = farthest from camera).
Objects with depth difference below 0.08 are likely on
  ...

── Condition: no_depth | 26 lines | 1295 chars ──
Objects in this scene:
  - Brown metal cube
  - Gray metal cube
  - Green metal sphere
  - Yellow metal cylinder
  - Yellow rubber sphere

Object overlaps detected: 2
  gray cube <-> green sphere first overlap at frame 99 (1 pixels)
  gree

---
# Cell 6 -- Parser and Scorer

In [10]:
# ── Cell 6 — Parser + Scorer ───────────────────────────────────────────────
import re

# ── Vocabulary normalization ───────────────────────────────────────────────
VOCAB_NORM = {
    'round': 'sphere', 'circular': 'sphere', 'ball': 'sphere',
    'square': 'cube', 'rectangular': 'cube', 'box': 'cube',
    'cylindrical': 'cylinder',
    'metallic': 'metal', 'shiny': 'metal', 'rubbery': 'rubber',
    'yeah': 'yes', 'yep': 'yes', 'nope': 'no'
}

def parse_response(raw_output, q_type):
    """
    Tiered parser for LLaVA output.
    Tier 1: clean extraction from ANSWER: field
    Tier 2: scan first line for valid tokens
    Tier 4: fallback -- empty string or empty list
    """
    text = raw_output.strip()

    # ── Extract ANSWER: field ──────────────────────────────────────────────
    answer_match = re.search(r'ANSWER:\s*(.+)', text, re.IGNORECASE)
    first_line   = answer_match.group(1).strip() if answer_match else text.split('\n')[0].strip()

    # ── Extract REASONING: field ───────────────────────────────────────────
    reasoning_match = re.search(r'REASONING:\s*(.+?)(?=ANSWER:|$)', text,
                                 re.IGNORECASE | re.DOTALL)
    reasoning = reasoning_match.group(1).strip() if reasoning_match else ""

    if q_type == 'descriptive':
        token = first_line.lower().strip('.,!?')
        token = VOCAB_NORM.get(token, token)
        tier  = 1 if answer_match else 2
        return {'parsed_answer': token, 'reasoning': reasoning, 'parse_tier': tier}

    elif q_type == 'predictive':
        letters = re.findall(r'\b([a-d])\b', first_line.lower())
        answer  = letters[0] if letters else ''
        tier    = 1 if answer_match else (2 if letters else 4)
        return {'parsed_answer': answer, 'reasoning': reasoning, 'parse_tier': tier}

    else:  # explanatory / counterfactual
        if re.search(r'\bnone\b', first_line, re.IGNORECASE):
            return {'parsed_answer': ['none'], 'reasoning': reasoning, 'parse_tier': 1}
        letters = sorted(set(re.findall(r'\b([a-d])\b', first_line.lower())))
        tier    = 1 if answer_match else (2 if letters else 4)
        return {'parsed_answer': letters if letters else [], 'reasoning': reasoning,
                'parse_tier': tier}


def get_ground_truth_letters(choices):
    return sorted([
        chr(97 + i)
        for i, c in enumerate(choices)
        if c.get('answer') == 'correct'
    ])


def score_question(parsed, q_type, question):
    """Score a single question. Returns dict with scoring results."""
    parsed_answer = parsed['parsed_answer']
    result        = {'q_type': q_type}

    if q_type == 'descriptive':
        ground_truth           = str(question.get('answer', '')).strip().lower()
        predicted              = str(parsed_answer).strip().lower()
        result['ground_truth'] = ground_truth
        result['predicted']    = predicted
        result['correct']      = (predicted == ground_truth)

    elif q_type == 'predictive':
        choices                = question.get('choices', [])
        gt_letters             = get_ground_truth_letters(choices)
        ground_truth           = gt_letters[0] if gt_letters else ''
        predicted              = str(parsed_answer).strip().lower()
        result['ground_truth'] = ground_truth
        result['predicted']    = predicted
        result['correct']      = (predicted == ground_truth)

    else:  # explanatory / counterfactual
        choices      = question.get('choices', [])
        gt_letters   = get_ground_truth_letters(choices)
        n_choices    = len(choices)
        pred_letters = [] if parsed_answer == ['none'] else parsed_answer

        # ── Per-option ────────────────────────────────────────────────────
        per_option_scores = []
        for i in range(n_choices):
            letter     = chr(97 + i)
            gt_correct = letter in gt_letters
            pd_correct = letter in pred_letters
            per_option_scores.append(gt_correct == pd_correct)

        per_option_acc = sum(per_option_scores) / n_choices if n_choices > 0 else 0.0

        # ── Per-question ──────────────────────────────────────────────────
        per_question_correct = (sorted(pred_letters) == sorted(gt_letters))

        # ── F1 ────────────────────────────────────────────────────────────
        if len(gt_letters) == 0 and len(pred_letters) == 0:
            f1 = 1.0
        elif len(gt_letters) == 0 or len(pred_letters) == 0:
            f1 = 0.0
        else:
            tp        = len(set(pred_letters) & set(gt_letters))
            precision = tp / len(pred_letters)
            recall    = tp / len(gt_letters)
            f1        = (2 * precision * recall / (precision + recall)
                         if (precision + recall) > 0 else 0.0)

        result['ground_truth']         = gt_letters
        result['predicted']            = pred_letters
        result['per_option_scores']    = per_option_scores
        result['per_option_acc']       = per_option_acc
        result['per_question_correct'] = per_question_correct
        result['f1']                   = round(f1, 4)

    return result


print("Parser and scorer defined.")

Parser and scorer defined.


---
# Cell 7 — Full Inference Loop

In [11]:
# ── Cell 7 — Full Inference Loop ───────────────────────────────────────────
import time
import os
from datetime import datetime
from PIL import Image
import torch

Q_TYPES   = ['descriptive', 'explanatory', 'predictive', 'counterfactual']
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
error_log = []

def build_prompt(q_type, question, context_str):
    """
    Build LLaVA prompt consistent with NB01 format.
    Context string injected after the domain pre-context, before the question.
    """
    pre_context = (
        "You are analyzing a single frame from a short video clip. "
        "The video depicts a physical scene with objects that may interact over time. "
        "You have been provided with structured physical measurements extracted from the full video. "
        "Use this information to answer the following question.\n\n"
        f"{context_str}\n\n"
    )

    if q_type == 'descriptive':
        prompt = (
            f"USER: <image>\n"
            f"{pre_context}"
            f"Question: {question['question']}\n"
            f"You must respond using exactly this format and no other format:\n"
            f"ANSWER: [single word or number]\n"
            f"REASONING: [2 sentences max explaining your observation]\n"
            f"Remember: use the ANSWER, REASONING format exactly.\n"
            f"ANSWER:"
        )

    elif q_type == 'predictive':
        choices_str = "\n".join([
            f"  {chr(97+i)}) {c['choice']}"
            for i, c in enumerate(question.get('choices', []))
        ])
        prompt = (
            f"USER: <image>\n"
            f"{pre_context}"
            f"Question: {question['question']}\n"
            f"Choices:\n{choices_str}\n"
            f"You must respond using exactly this format and no other format:\n"
            f"ANSWER: [single letter only, e.g. a]\n"
            f"REASONING: [2 sentences max explaining your observation]\n"
            f"Remember: use the ANSWER, REASONING format exactly.\n"
            f"ANSWER:"
        )

    else:  # explanatory / counterfactual
        choices_str = "\n".join([
            f"  {chr(97+i)}) {c['choice']}"
            for i, c in enumerate(question.get('choices', []))
        ])
        prompt = (
            f"USER: <image>\n"
            f"{pre_context}"
            f"Question: {question['question']}\n"
            f"Choices:\n{choices_str}\n"
            f"You must respond using exactly this format and no other format:\n"
            f"ANSWER: [one or more letters comma-separated e.g. a, b -- or 'None' if none apply]\n"
            f"REASONING: [2 sentences max explaining your observation]\n"
            f"Remember: use the ANSWER, REASONING format exactly.\n"
            f"ANSWER:"
        )

    return prompt

def run_llava(image, prompt):
    """Run LLaVA on one image + prompt. Returns raw output string."""
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )
    # Decode only newly generated tokens
    input_len  = inputs['input_ids'].shape[1]
    new_tokens = output_ids[0][input_len:]
    return processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def process_clip(vid_id, condition):
    """
    Run LLaVA on all questions for one clip under one ablation condition.
    Returns list of question result dicts.
    """
    kf_path    = os.path.join(KF_DIR, f"frame_{vid_id:05d}.jpg")
    image      = Image.open(kf_path).convert("RGB")
    qa_entry   = qa_by_id.get(vid_id, {})
    questions  = qa_entry.get('questions', [])
    ctx_str    = assemble_context(vid_id, condition)
    results    = []

    for q in questions:
        q_type = q.get('question_type', '')
        if q_type not in Q_TYPES:
            continue
        try:
            prompt     = build_prompt(q_type, q, ctx_str)
            raw_output = run_llava(image, prompt)
            parsed     = parse_response(raw_output, q_type)
            scored     = score_question(parsed, q_type, q)

            q_result = {
                'question_id'    : q.get('question_id'),
                'question_type'  : q_type,
                'question'       : q.get('question'),
                'condition'      : condition,
                'context_string' : ctx_str,
                'raw_output'     : raw_output,
                'parsed_answer'  : parsed['parsed_answer'],
                'reasoning'      : parsed['reasoning'],
                'parse_tier'     : parsed['parse_tier'],
            }

            # ── Merge scoring fields ───────────────────────────────────────
            q_result.update({k: v for k, v in scored.items() if k != 'q_type'})

            if q_type in ('explanatory', 'counterfactual'):
                q_result['choices'] = [
                    {'choice': c['choice'], 'answer': c.get('answer', '')}
                    for c in q.get('choices', [])
                ]

        except Exception as e:
            err_msg = str(e)
            error_log.append({
                'vid_id': vid_id, 'condition': condition,
                'question_id': q.get('question_id'), 'error': err_msg
            })
            q_result = {
                'question_id'  : q.get('question_id'),
                'question_type': q_type,
                'condition'    : condition,
                'correct'      : None,
                'error'        : err_msg
            }

        results.append(q_result)

    return results, ctx_str

- uncomment only if needed

In [12]:
# # ── Delete all enhanced JSONs to rerun with new prompt ────────────────────
# import os

# for condition in ABLATION_CONDITIONS:
#     folder = OUT_ENHANCED[condition]
#     for vid_id in SELECTED_IDS:
#         path = os.path.join(folder, f"enhanced_{vid_id:05d}.json")
#         if os.path.exists(path):
#             os.remove(path)
#             print(f"Deleted: {path}")

# print("Done.")

In [13]:
# ── Main loop ──────────────────────────────────────────────────────────────
for vid_id in SELECTED_IDS:
    kf_path = os.path.join(KF_DIR, f"frame_{vid_id:05d}.jpg")
    if not os.path.exists(kf_path):
        print(f"Clip {vid_id}: keyframe missing, skipping.")
        continue

    for condition in ABLATION_CONDITIONS:
        # out_path = os.path.join(OUT_DIR, f"enhanced_{condition}_{vid_id:05d}.json")
        out_path = os.path.join(OUT_ENHANCED[condition], f"enhanced_{vid_id:05d}.json")

        if os.path.exists(out_path):
            print(f"Clip {vid_id} [{condition}]: already done, skipping.")
            continue

        print(f"\nClip {vid_id} [{condition}]...")
        clip_start = time.time()

        clip_results, ctx_str = process_clip(vid_id, condition)

        clip_runtime = round(time.time() - clip_start, 2)
        clip_output  = {
            'clip_id'       : vid_id,
            'condition'     : condition,
            'runtime_sec'   : clip_runtime,
            'context_string': ctx_str,
            'questions'     : clip_results
        }

        out_path = os.path.join(OUT_ENHANCED[condition], f"enhanced_{vid_id:05d}.json")
        with open(out_path, 'w') as f:
            json.dump(clip_output, f, indent=2)

        # ── Per-clip summary ───────────────────────────────────────────────
        type_counts = {}
        for qr in clip_results:
            qt = qr['question_type']
            if qt not in type_counts:
                type_counts[qt] = {'correct': 0, 'total': 0}
            if qt in ('descriptive', 'predictive'):
                if qr.get('correct') is not None:
                    type_counts[qt]['total']   += 1
                    type_counts[qt]['correct'] += int(qr['correct'])
            else:
                if qr.get('per_question_correct') is not None:
                    type_counts[qt]['total']   += 1
                    type_counts[qt]['correct'] += int(qr['per_question_correct'])

        summary = " | ".join([
            f"{qt[:4]}: {type_counts[qt]['correct']}/{type_counts[qt]['total']}"
            for qt in Q_TYPES if qt in type_counts
        ])
        print(f"  Done in {clip_runtime:.1f}s | {summary}")
        print(f"  Saved -> {out_path}")


Clip 10001 [full]...
  Done in 34.0s | desc: 3/11 | expl: 0/2 | pred: 1/1 | coun: 0/2
  Saved -> /content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project/outputs/stage4_llava/full/enhanced_10001.json

Clip 10001 [no_sam2]...
  Done in 36.3s | desc: 3/11 | expl: 0/2 | pred: 1/1 | coun: 0/2
  Saved -> /content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project/outputs/stage4_llava/no_sam2/enhanced_10001.json

Clip 10001 [no_depth]...
  Done in 30.6s | desc: 3/11 | expl: 0/2 | pred: 1/1 | coun: 0/2
  Saved -> /content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project/outputs/stage4_llava/no_depth/enhanced_10001.json

Clip 10001 [no_raft]...
  Done in 23.8s | desc: 4/11 | expl: 0/2 | pred: 1/1 | coun: 0/2
  Saved -> /content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project/outputs/stage4_llava/no_raft/enhanced_10001.json

Clip 10002 [full]...
  Done in 36.7s | desc: 5/11 | expl: 0/2 | coun: 1/2
  Saved -> /conte

In [14]:
# ── Save error log ─────────────────────────────────────────────────────────
if error_log:
    err_path = os.path.join(OUT_DIR, f"enhanced_errors_{timestamp}.json")
    with open(err_path, 'w') as f:
        json.dump(error_log, f, indent=2)
    print(f"\nError log saved -> {err_path} ({len(error_log)} errors)")
else:
    print("\nNo errors encountered.")

print("\nInference loop complete.")


No errors encountered.

Inference loop complete.


In [16]:
# ── Sanity Check — One multi-select question across all 4 conditions ───────
import json, os

vid_id        = 10001
q_type_target = 'explanatory'

print(f"── Sanity Check: Clip {vid_id} | first '{q_type_target}' question ──\n")

for condition in ABLATION_CONDITIONS:
    path = os.path.join(OUT_ENHANCED[condition], f"enhanced_{vid_id:05d}.json")
    with open(path, 'r') as f:
        data = json.load(f)

    q = next((q for q in data['questions'] if q['question_type'] == q_type_target), None)
    if q is None:
        print(f"[{condition}] No {q_type_target} question found.")
        continue

    ctx_str = data['context_string']
    prompt  = build_prompt(q['question_type'], q, ctx_str)

    print(f"{'='*70}")
    print(f"CONDITION: {condition}")
    print(f"{'='*70}")
    print(f"── Full Prompt ───────────────────────────────────────────────────")
    print(prompt)
    print(f"\n── Raw LLaVA Output ──────────────────────────────────────────────")
    print(q['raw_output'])
    print(f"\n── Scoring ───────────────────────────────────────────────────────")
    print(f"  Question     : {q['question']}")
    print(f"  Choices      : {[(c['choice'], c['answer']) for c in q.get('choices', [])]}")
    print(f"  Ground truth : {q.get('ground_truth')}")
    print(f"  Predicted    : {q.get('predicted')}")
    print(f"  Reasoning    : {q.get('reasoning', '')}"),
    print(f"  Per-Q correct: {q.get('per_question_correct')}")
    print(f"  F1           : {q.get('f1')}")
    print()

── Sanity Check: Clip 10001 | first 'explanatory' question ──

CONDITION: full
── Full Prompt ───────────────────────────────────────────────────
USER: <image>
You are analyzing a single frame from a short video clip. The video depicts a physical scene with objects that may interact over time. You have been provided with structured physical measurements extracted from the full video. Use this information to answer the following question.

Objects in this scene:
  - Brown metal cube
  - Gray metal cube
  - Green metal sphere
  - Yellow metal cylinder
  - Yellow rubber sphere

Object overlaps detected: 2
  gray cube <-> green sphere first overlap at frame 99 (1 pixels)
  green sphere <-> yellow cylinder first overlap at frame 92 (1 pixels)

Depth information (relative estimates within each frame,
0 = closest to camera, 1 = farthest from camera).
Objects with depth difference below 0.08 are likely on the same collision plane.

Depth at spatial overlap events:
  - Gray metal cube and green

---
# Cell 8 — Summary Table + Save CSVs

In [17]:
# ── Cell 8 — Summary Table + Save CSVs ────────────────────────────────────
import csv

Q_TYPES = ['descriptive', 'explanatory', 'predictive', 'counterfactual']

def aggregate_results(condition):
    """
    Load all enhanced JSONs for a condition and aggregate scores.
    Returns metrics dict per question type.
    """
    agg = {
        qt: {
            'correct': 0, 'total': 0,
            'per_opt_correct': 0, 'per_opt_total': 0,
            'per_q_correct': 0, 'per_q_total': 0,
            'f1_sum': 0.0, 'f1_count': 0
        }
        for qt in Q_TYPES
    }

    folder = OUT_ENHANCED[condition]
    for fname in sorted(os.listdir(folder)):
        if not (fname.startswith('enhanced_') and fname.endswith('.json')):
            continue
        with open(os.path.join(folder, fname), 'r') as f:
            clip = json.load(f)

        for qr in clip.get('questions', []):
            qt = qr.get('question_type')
            if qt not in agg:
                continue

            if qt in ('descriptive', 'predictive'):
                if qr.get('correct') is None:
                    continue
                agg[qt]['total']   += 1
                agg[qt]['correct'] += int(qr['correct'])

            else:
                if qr.get('per_question_correct') is None:
                    continue
                per_opt = qr.get('per_option_scores', [])
                agg[qt]['per_opt_total']   += len(per_opt)
                agg[qt]['per_opt_correct'] += sum(per_opt)
                agg[qt]['per_q_total']     += 1
                agg[qt]['per_q_correct']   += int(qr['per_question_correct'])
                agg[qt]['f1_sum']          += qr.get('f1', 0.0)
                agg[qt]['f1_count']        += 1

    # ── Compute final metrics ──────────────────────────────────────────────
    metrics = {}
    for qt in Q_TYPES:
        a = agg[qt]
        if qt in ('descriptive', 'predictive'):
            per_q_acc = a['correct'] / a['total'] if a['total'] > 0 else 0.0
            metrics[qt] = {
                'total'      : a['total'],
                'per_q_acc'  : round(per_q_acc, 4),
                'per_opt_acc': None,
                'f1'         : None
            }
        else:
            per_q_acc   = a['per_q_correct']  / a['per_q_total']   if a['per_q_total']   > 0 else 0.0
            per_opt_acc = a['per_opt_correct'] / a['per_opt_total'] if a['per_opt_total'] > 0 else 0.0
            avg_f1      = a['f1_sum'] / a['f1_count']               if a['f1_count']      > 0 else 0.0
            metrics[qt] = {
                'total'      : a['per_q_total'],
                'per_q_acc'  : round(per_q_acc, 4),
                'per_opt_acc': round(per_opt_acc, 4),
                'f1'         : round(avg_f1, 4)
            }

    return metrics

In [18]:
# ── Aggregate all conditions ───────────────────────────────────────────────
all_metrics = {cond: aggregate_results(cond) for cond in ABLATION_CONDITIONS}

In [19]:
# ── Print side-by-side summary table ──────────────────────────────────────
print(f"{'='*75}")
print(f"  ENHANCED RESULTS -- {len(SELECTED_IDS)} clips")
print(f"{'='*75}")
print(f"  {'Type':<16} {'Condition':<12} {'Total Q':>8} {'Per-Q Acc':>10} {'Per-Opt Acc':>12} {'F1':>8}")
print(f"  {'-'*74}")

for qt in Q_TYPES:
    for cond in ABLATION_CONDITIONS:
        m = all_metrics[cond][qt]
        per_opt = f"{m['per_opt_acc']:.4f}" if m['per_opt_acc'] is not None else '--'
        f1      = f"{m['f1']:.4f}"          if m['f1']         is not None else '--'
        print(f"  {qt:<16} {cond:<12} {m['total']:>8} {m['per_q_acc']:>10.4f} {per_opt:>12} {f1:>8}")
    print(f"  {'-'*74}")

# ── Save one CSV per condition ─────────────────────────────────────────────
for cond in ABLATION_CONDITIONS:
    csv_path = os.path.join(ANALYSIS_DIR, f"enhanced_{cond}_summary.csv")
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'condition', 'question_type', 'total',
            'per_q_acc', 'per_opt_acc', 'f1'
        ])
        writer.writeheader()
        for qt in Q_TYPES:
            m = all_metrics[cond][qt]
            writer.writerow({
                'condition'    : cond,
                'question_type': qt,
                'total'        : m['total'],
                'per_q_acc'    : m['per_q_acc'],
                'per_opt_acc'  : m['per_opt_acc'],
                'f1'           : m['f1']
            })
    print(f"Saved -> {csv_path}")

print("\nCell 8 complete.")

  ENHANCED RESULTS -- 70 clips
  Type             Condition     Total Q  Per-Q Acc  Per-Opt Acc       F1
  --------------------------------------------------------------------------
  descriptive      full              770     0.2909           --       --
  descriptive      no_sam2           770     0.2896           --       --
  descriptive      no_depth          770     0.2857           --       --
  descriptive      no_raft           770     0.2961           --       --
  --------------------------------------------------------------------------
  explanatory      full              129     0.0698       0.5553   0.4273
  explanatory      no_sam2           129     0.0620       0.5445   0.4162
  explanatory      no_depth          129     0.0543       0.5315   0.3912
  explanatory      no_raft           129     0.0620       0.5336   0.4258
  --------------------------------------------------------------------------
  predictive       full               50     0.4800           --       -

In [20]:
# ── Check predictive question across all conditions for clip 10001 ─────────
vid_id = 10001

for condition in ABLATION_CONDITIONS:
    path = os.path.join(OUT_ENHANCED[condition], f"enhanced_{vid_id:05d}.json")
    with open(path, 'r') as f:
        data = json.load(f)

    q = next((q for q in data['questions'] if q['question_type'] == 'predictive'), None)
    if q is None:
        print(f"[{condition}] No predictive question found.")
        continue

    print(f"[{condition}]")
    print(f"  Question     : {q['question']}")
    print(f"  Choices      : {[(c['choice'], c['answer']) for c in q.get('choices', [])]}")
    print(f"  Ground truth : {q.get('ground_truth')}")
    print(f"  Predicted    : {q.get('predicted')}")
    print(f"  Correct      : {q.get('correct')}")
    print(f"  Raw output   : {q.get('raw_output')}")
    print()

[full]
  Question     : What will happen next?
  Choices      : []
  Ground truth : a
  Predicted    : a
  Correct      : True
  Raw output   : a
REASONING: The rubber object is moving towards the gray object, which is closer to the camera. The rubber object is likely to collide with the gray object.

[no_sam2]
  Question     : What will happen next?
  Choices      : []
  Ground truth : a
  Predicted    : a
  Correct      : True
  Raw output   : a
REASONING: The rubber object is moving towards the gray object, which is moving upward. The two objects are likely to collide.

[no_depth]
  Question     : What will happen next?
  Choices      : []
  Ground truth : a
  Predicted    : a
  Correct      : True
  Raw output   : a

[no_raft]
  Question     : What will happen next?
  Choices      : []
  Ground truth : a
  Predicted    : a
  Correct      : True
  Raw output   : a



---
# Cell 9 — Drive Space Check + Completion Summary

In [21]:
# ── Cell 9 — Drive Space Check + Completion Summary ───────────────────────

print("── Stage 4 Enhanced Output Summary ───────────────────────────────────")
total_bytes = 0
for condition in ABLATION_CONDITIONS:
    folder = OUT_ENHANCED[condition]
    files  = [f for f in os.listdir(folder) if f.endswith('.json')]
    size   = sum(os.path.getsize(os.path.join(folder, f)) for f in files)
    total_bytes += size
    print(f"  {condition:<12} : {len(files)} files | {size/1e3:.1f} KB")

print(f"\n  Total size   : {total_bytes/1e6:.2f} MB")
print(f"  Output dir   : {OUT_DIR}")
print()
print("── CSVs saved ────────────────────────────────────────────────────────")
for cond in ABLATION_CONDITIONS:
    csv_path = os.path.join(ANALYSIS_DIR, f"enhanced_{cond}_summary.csv")
    if os.path.exists(csv_path):
        print(f"  {csv_path}")

print()
print("Notebook 03 complete. Proceed to 04_analysis.ipynb.")

── Stage 4 Enhanced Output Summary ───────────────────────────────────
  full         : 70 files | 3369.5 KB
  no_sam2      : 70 files | 3122.9 KB
  no_depth     : 70 files | 2461.8 KB
  no_raft      : 70 files | 2124.5 KB

  Total size   : 11.08 MB
  Output dir   : /content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project/outputs/stage4_llava

── CSVs saved ────────────────────────────────────────────────────────
  /content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project/analysis/enhanced_full_summary.csv
  /content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project/analysis/enhanced_no_sam2_summary.csv
  /content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project/analysis/enhanced_no_depth_summary.csv
  /content/drive/MyDrive/NEU - MS CS/4_SEM (Jan 2026)/CV/Projects/final_project/analysis/enhanced_no_raft_summary.csv

Notebook 03 complete. Proceed to 04_analysis.ipynb.


---